In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import time
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from clearml import Task
from dotenv import load_dotenv
from torch import nn
from torch.utils.data import DataLoader

from mglyph_ml.dataset.glyph_dataset import GlyphDataset
from mglyph_ml.dataset.utils import show_datasets

In [ ]:
task_name = "EXP-regression-2.0.0"
task_tag = "exp-regression-gen3"
dataset_path = Path("data/uni.mglyph")
seed = 420
max_epochs = 40
offline = False
batch_size = 64
data_loader_num_workers = 16
sample_count = 5000

In [ ]:
Task.set_offline(offline)
task: Task = Task.init(project_name="mglyph-ml", task_name=task_name)
task.add_tags(task_tag)
load_dotenv()

task.connect({
    "dataset_path": str(dataset_path),
    "seed": seed,
    "max_epochs": max_epochs,
    "batch_size": batch_size,
    "data_loader_num_workers": data_loader_num_workers,
    "sample_count": sample_count,
})

In [ ]:
loaded_split_0 = load_images_and_labels(
    dataset_path=dataset_path,
    split="0",
    desired_size=(64, 64),
    shuffle=True,
    seed=seed,
)

train_limit = min(sample_count, len(loaded_split_0.images))
images_train = loaded_split_0.images[:train_limit]
labels_train = loaded_split_0.labels[:train_limit]
images_overall = images_train
labels_overall = labels_train

In [ ]:
affine = A.Affine(
    rotate=(-3, 3),
    translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
    fit_output=False,
    keep_ratio=True,
    border_mode=cv2.BORDER_CONSTANT,
    fill=255,
    p=1.0,
)
normalize = A.Normalize(mean=0.0, std=1.0, max_pixel_value=255.0)
to_tensor = ToTensorV2()
pipeline = A.Compose([affine, normalize, to_tensor], seed=seed)
normalize_pipeline = A.Compose([normalize, to_tensor])


def affine_and_normalize(image: np.ndarray) -> torch.Tensor:
    return pipeline(image=image)["image"]


def just_normalize(image: np.ndarray) -> torch.Tensor:
    return normalize_pipeline(image=image)["image"]


dataset_train = GlyphDataset(
    name="Training",
    images=images_train,
    labels=labels_train,
    transform=just_normalize,
)
dataset_overall = GlyphDataset(
    name="Overall",
    images=images_overall,
    labels=labels_overall,
    transform=just_normalize,
)

print(f"train dataset size: {len(dataset_train)}")

In [ ]:
show_datasets(dataset_train, dataset_overall)

In [ ]:
from mglyph_ml.nn.glyph_regressor_gen3 import GlyphRegressorGen3

device = os.environ["MGML_DEVICE"]
print(f"Training device: {device}")
model = GlyphRegressorGen3(image_resolution=(64, 64))

learning_rate = 2e-4
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

generator = torch.Generator().manual_seed(seed)

data_loader_train = DataLoader(
    dataset_train,
    batch_size=batch_size,
    num_workers=data_loader_num_workers,
    pin_memory=True,
    shuffle=True,
    generator=generator,
)

data_loader_train_eval = DataLoader(
    dataset_train,
    batch_size=batch_size,
    num_workers=data_loader_num_workers,
    pin_memory=True,
    shuffle=False,
)

data_loader_overall = DataLoader(
    dataset_overall,
    batch_size=batch_size,
    num_workers=data_loader_num_workers,
    pin_memory=True,
    shuffle=False,
)

In [ ]:
model.to(device)
for epoch in range(1, max_epochs + 1):
    model.train()
    epoch_start_time = time.time()
    running_train_loss = 0.0
    running_error_train = 0.0
    num_batches_train = 0

    for index, data in enumerate(data_loader_train):
        inputs, labels = data

        if index == 0:
            print(inputs.shape)

        inputs: torch.Tensor = inputs.to(device)
        labels: torch.Tensor = labels.to(device)

        inputs = inputs.float()
        labels = labels.float().view(-1)

        optimizer.zero_grad()
        preds: torch.Tensor = model(inputs).view(-1)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()

        error = torch.mean(torch.abs(preds - labels)).item()

        running_train_loss += loss.item()
        running_error_train += error
        num_batches_train += 1
    
    avg_train_loss = running_train_loss / num_batches_train
    avg_train_error = running_error_train / num_batches_train
    avg_train_error_x = avg_train_error * 100.0

    # # Set model to evaluation mode and evaluate with fixed weights.
    # model.eval()

    # def evaluate_loader(data_loader: DataLoader) -> tuple[float, float]:
    #     running_loss = 0.0
    #     running_error = 0.0
    #     num_batches = 0

    #     with torch.no_grad():
    #         for inputs, labels in data_loader:
    #             inputs = inputs.to(device).float()
    #             labels = labels.to(device).float().view(-1)

    #             preds = model(inputs).view(-1)
    #             loss = criterion(preds, labels)
    #             error = torch.mean(torch.abs(preds - labels)).item()

    #             running_loss += loss.item()
    #             running_error += error
    #             num_batches += 1

    #     avg_loss = running_loss / num_batches
    #     avg_error = running_error / num_batches
    #     return avg_loss, avg_error

    # # True train-set eval with same final weights as overall eval.
    # avg_train_loss_eval, avg_train_error_eval = evaluate_loader(data_loader_train_eval)
    # avg_train_error_x_eval = avg_train_error_eval * 100.0

    # # Overall evaluation (full 0-100 range).
    # avg_overall_loss, avg_overall_error = evaluate_loader(data_loader_overall)
    # avg_overall_error_x = avg_overall_error * 100.0

    epoch_time = time.time() - epoch_start_time

    # Log metrics to ClearML
    task.logger.report_scalar(title="Loss", series="Train", value=avg_train_loss, iteration=epoch)
    task.logger.report_scalar(title="MAE (x-space)", series="Train", value=avg_train_error_x, iteration=epoch)

    # task.logger.report_scalar(title="Loss", series="Overall", value=avg_overall_loss, iteration=epoch)
    # task.logger.report_scalar(title="MAE (x-space)", series="Overall", value=avg_overall_error_x, iteration=epoch)

    task.logger.report_scalar(title="Timing", series="Epoch Duration (s)", value=epoch_time, iteration=epoch)

    # Print epoch summary
    print("=" * 80)
    print(f"Epoch {epoch}/{max_epochs}")
    print("-" * 80)
    print(f"  Train Loss: {avg_train_loss:.6f}  |  Train MAE (x): {avg_train_error_x:.4f}  |  ")
    # Overall MAE (x): {avg_overall_error_x:.4f}")
    print(f"  Epoch Time: {epoch_time:.2f}s")
    print("=" * 80)

In [ ]:
import random as _py_random

model.to("cpu")

sample_indices = _py_random.sample(range(len(dataset_overall)), min(1000, len(dataset_overall)))

model.eval()
actuals = []
predictions = []

with torch.no_grad():
    for idx in sample_indices:
        img_tensor, label = dataset_overall[idx]
        img_tensor = img_tensor.unsqueeze(0).float()
        pred = model(img_tensor).view(-1).item()
        actuals.append(label * 100.0)
        predictions.append(pred * 100.0)

actuals = np.array(actuals)
predictions = np.array(predictions)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(actuals, predictions, alpha=0.6, s=20, label="Samples")

lo = min(actuals.min(), predictions.min())
hi = max(actuals.max(), predictions.max())
ax.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="Perfect prediction")

ax.set_xlabel("Actual label (raw x)")
ax.set_ylabel("Predicted label (raw x)")
ax.set_title("Actual vs Predicted (Regression)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import random as _py_random

model.eval()
mse_criterion = nn.MSELoss()

n_samples = min(1000, len(dataset_train))
sample_indices = _py_random.sample(range(len(dataset_train)), n_samples)

x_vals = []
losses_per_sample = []

with torch.no_grad():
    for idx in sample_indices:
        img_tensor, label = dataset_train[idx]
        img_tensor = img_tensor.unsqueeze(0).float()
        label_tensor = torch.tensor([label], dtype=torch.float32)

        pred = model(img_tensor).view(-1)
        loss = mse_criterion(pred, label_tensor)

        x_vals.append(label * 100.0)
        losses_per_sample.append(loss.item())

x_vals = np.array(x_vals)
losses_per_sample = np.array(losses_per_sample)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_vals, losses_per_sample, alpha=0.5, s=12)
ax.set_xlabel("Actual label (raw x)")
ax.set_ylabel("Loss (MSE)")
ax.set_title(f"Loss vs. x (random {n_samples} training samples)")
plt.tight_layout()
plt.show()

In [ ]:
task.flush(wait_for_uploads=True)
task.close()